Check whether this privacy policy existed in the Wayback Machine during the past time period. 

Traverse the previously saved file (found_privacy_policy_url.txt), iterate over the privacy policy URLs, search each URL's history in the Wayback Machine, and check whether there are records within one year before or after October 2024.

Traverse the previously saved file 1122apk/1122apk_privacy_policy_url/extracted_pp_urls.json, where the key is apkname and the value is the corresponding privacy policy URL. Read each APK's privacy policy URL, search the URL's history in the Wayback Machine, and check whether there are records within one year before or after October 2024.

If a privacy policy URL for the same apkname but a different version already exists, skip it.

Traverse extracted_pp_urls.json
2️⃣ Query Wayback
3️⃣ Find the snapshot closest to 2024-10-15
4️⃣ Write the results to CSV (intermediate results)
5️⃣ If a snapshot exists → download MD/TXT instead of HTML

In [ ]:
# {f_position}\{s_position}
f_position = "AA2_second_batch"
s_position = "AA6_sisth_100_batch" # AA6_sisth_100_batch, AA7_seventh_100_batch # AA3_third_100_batch, AA4_forth_100_batch, AA5_fifth_100_batch

In [167]:
import os
import json
import csv
import time
import random
import requests
from urllib.parse import urlparse
import re
from html import unescape
import csv
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from requests.exceptions import ReadTimeout, RequestException
from pathlib import Path

In [168]:
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA2_second_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA3_third_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA4_forth_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA5_fifth_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA6_sisth_74_batch\batch_privacy_results.csv
INPUT_JSON = rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_url\{f_position}\{s_position}\batch_privacy_results.csv" #r"1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA2_second_100_batch\final_batch_privacy_results.csv" #r"1122apk/1122apk_privacy_policy_url/out/extracted_pp_urls.json"

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\waybackmachine
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\waybackmachine
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\waybackmachine
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\waybackmachine
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\waybackmachine
OUTPUT_DIR = rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\waybackmachine" #r"1122apk/1122apk_privacy_policy_url/pp_html_2024_10"

output = Path(OUTPUT_DIR)
output.mkdir(parents=True, exist_ok=True)

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\waybackmachine\pp_wayback_result.csv
CSV_FILE = rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\waybackmachine\pp_wayback_result.csv" #"1122apk/1122apk_privacy_policy_url/out/pp_wayback_result.csv"

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_JSON, OUTPUT_DIR, CSV_FILE

('1122apk\\1122apk_privacy_policy_url\\out_openai_pp_url\\AA2_second_batch\\AA5_fifth_100_batch\\batch_privacy_results.csv',
 '1122apk\\1122apk_privacy_policy_url\\pp_md_2024_10\\AA2_second_batch\\AA5_fifth_100_batch\\waybackmachine',
 '1122apk\\1122apk_privacy_policy_url\\pp_md_2024_10\\AA2_second_batch\\AA5_fifth_100_batch\\waybackmachine\\pp_wayback_result.csv')

In [169]:
FROM_TS = "20231001"
TO_TS   = "20251031"
TARGET_TS = "20241015000000"   # Used to find the record closest to 2024-10-15

CDX_API = "https://web.archive.org/cdx/search/cdx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

In [170]:
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"])
)
adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
session.mount("http://", adapter)
session.mount("https://", adapter)

def safe_get(url, params=None, timeout=(10, 60), tries=3):
    for t in range(tries):
        try:
            return session.get(url, params=params, timeout=timeout)
        except ReadTimeout:
            if t == tries - 1:
                raise
            time.sleep((1.5 ** t) + random.uniform(0.2, 0.8))
        except RequestException:
            if t == tries - 1:
                raise
            time.sleep((1.5 ** t) + random.uniform(0.2, 0.8))


def slugify_heading_en(s: str) -> str:
    s = s.strip().lower()
    # Keep only English letters, digits, spaces, and hyphens
    s = re.sub(r"[^a-z0-9\s-]", "", s)
    s = re.sub(r"\s+", "-", s)
    s = re.sub(r"-+", "-", s).strip("-")
    return s

def build_toc_en(md_text: str) -> str:
    toc_lines = []
    for line in md_text.splitlines():
        m = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)
        if not m:
            continue
        level = len(m.group(1))
        title = m.group(2).strip()
        anchor = slugify_heading_en(title)
        if not anchor:
            continue
        indent = "  " * (level - 1)
        toc_lines.append(f"{indent}- [{title}](#{anchor})")

    if not toc_lines:
        return ""
    return "## Table of Contents\n\n" + "\n".join(toc_lines) + "\n\n"

In [171]:
def normalize_url(url):
    if url is None or pd.isna(url):
        return None
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    return url


def ts_distance(a, b):
    return abs(int(a) - int(b))


def find_closest_snapshot(url):

    params = {
        "url": url,
        "from": FROM_TS,
        "to": TO_TS,
        "output": "json",
        "fl": "timestamp,original",
        "collapse": "digest",
        "filter": "statuscode:200"
    }

    try:
        r = safe_get(CDX_API, params=params, timeout=(10, 60), tries=3)
    except Exception:
        return None

    if r.status_code != 200:
        return None

    data = r.json()

    if len(data) <= 1:
        return None

    header = data[0]
    rows = data[1:]

    records = []

    for row in rows:
        item = dict(zip(header, row))
        records.append(item)

    closest = min(records, key=lambda x: ts_distance(x["timestamp"], TARGET_TS))

    return closest



def html_to_text(html):
    # Remove script/style blocks
    html = re.sub(r"(?is)<script.*?>.*?</script>", " ", html)
    html = re.sub(r"(?is)<style.*?>.*?</style>", " ", html)

    # Add line breaks for common block tags to avoid merging everything together
    html = re.sub(r"(?i)</p>|<br\s*/?>|</div>|</li>|</tr>|</h[1-6]>", "\n", html)
    html = re.sub(r"(?i)<li[^>]*>", "- ", html)

    # Remove all tags
    text = re.sub(r"(?s)<[^>]+>", " ", html)

    # Unescape HTML entities
    text = unescape(text)

    # Clean up redundant whitespace
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    return text.strip()


def download_snapshot_1(apk_name, snapshot):

    ts = snapshot["timestamp"]
    url = snapshot["original"]

    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    out_file = os.path.join(OUTPUT_DIR, apk_name + ".html")

    try:
        r = session.get(wayback_url, timeout=60)

        if r.status_code != 200:
            return False

        with open(out_file, "wb") as f:
            f.write(r.content)

        return True

    except Exception:
        return False
    
def download_snapshot(apk_name, version, snapshot):
    ts = snapshot["timestamp"]
    original = snapshot["original"]

    # Use id_ to retrieve the archived original content and reduce interference from Wayback's top navigation
    wayback_url = f"https://web.archive.org/web/{ts}id_/{original}"

    out_file = os.path.join(OUTPUT_DIR, f"{apk_name}_{version}.md")

    try:
        r = safe_get(wayback_url, timeout=(10, 90), tries=3)
        if r.status_code != 200:
            return False

        r.encoding = r.apparent_encoding or r.encoding

        # First try a real HTML-to-Markdown conversion; fall back to the existing plain-text extraction if it fails
        md_text = None
        try:
            from markdownify import markdownify as md
            # md_text = md(r.text, heading_style="ATX")
        except Exception:
            # md_text = html_to_text(r.text)
            print(f"Failed to convert HTML to Markdown for {apk_name} version {version}. Saving as plain text.")

        # toc = build_toc_en(md_text)
        # final_md = (toc + md_text).strip()

        # with open(out_file, "w", encoding="utf-8") as f:
        #     f.write(md_text) #f.write(final_md) 

        return True
    except Exception:
        return False
    
def download_snapshot_txt(apk_name, version, snapshot):

    ts = snapshot["timestamp"]
    url = snapshot["original"]

    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    out_file = os.path.join(OUTPUT_DIR, f"{apk_name}_{version}.txt")

    try:
        r = session.get(wayback_url, timeout=60)

        if r.status_code != 200:
            return False

        # Decode using the encoding detected by requests whenever possible
        r.encoding = r.apparent_encoding or r.encoding
        text = html_to_text(r.text)

        with open(out_file, "w", encoding="utf-8") as f:
            f.write(text)

        return True

    except Exception:
        return False


def append_csv(row):

    file_exists = os.path.exists(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "apk_name",
                "version",
                "pp_url",
                "timestamp",
                "wayback_url",
                "has_snapshot",
                "downloaded"
            ]
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

In [172]:
# # with open(INPUT_JSON, "r", encoding="utf-8-sig", newline="") as f:
# #     reader = csv.DictReader(f)
# #     data = list(reader)
# data = pd.read_csv(INPUT_JSON, encoding="utf-8-sig")
# data.head(), data.shape, len(data)

# for i, row in data.iterrows():
#     apk_name = row["apk_name"]
#     pp_url = row["privacy_url"]

#     print(i, apk_name, pp_url)

In [173]:
data = pd.read_csv(INPUT_JSON, encoding="utf-8-sig")
data.head(), data.shape, len(data)

total = len(data)

print("Total:", total)

replicated_apk = {} 

for i, row in data.iterrows():
    apk_name = row["apk_name"]
    version = row["version"]
    url = row["privacy_url"]
    print(i, apk_name, version, url)

    if not url:
        url = None
    else:
        url = url

    url = normalize_url(url)

    # print(f"[{i}/{total}] {apk_name}")

    if not url:

        append_csv({
            "apk_name": apk_name,
            "version": version,
            "pp_url": "",
            "timestamp": "",
            "wayback_url": "",
            "has_snapshot": False,
            "downloaded": False
        })

        continue

    snapshot = find_closest_snapshot(url)

    if not snapshot:

        print("   no snapshot")

        append_csv({
            "apk_name": apk_name,
            "version": version,
            "pp_url": url,
            "timestamp": "",
            "wayback_url": "",
            "has_snapshot": False,
            "downloaded": False
        })

        continue

    ts = snapshot["timestamp"]
    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    print("   snapshot:", ts)

    if apk_name in replicated_apk:
        if replicated_apk[apk_name] == url:
            print("url already replicated, skipping download.")
            downloaded = True  # Mark as downloaded because we already have a snapshot for this URL
    else:
        downloaded = download_snapshot(apk_name, version, snapshot)

        if downloaded:
            replicated_apk[apk_name] = url

    append_csv({
        "apk_name": apk_name,
        "version": version,
        "pp_url": url,
        "timestamp": ts,
        "wayback_url": wayback_url,
        "has_snapshot": True,
        "downloaded": downloaded
    })

    time.sleep(1)

print("Done.", CSV_FILE)

Total: 98
0 com.S2.WaStickerApps 33 nan
1 com.S2.WaStickerApps 34 nan
2 com.Slbazer.Craftsman4 972000073 nan
3 com.Slbazer.Craftsman4 972000074 nan
4 com.rubygames.splashyjump 3902 nan
5 com.rubygames.splashyjump 3903 nan
6 com.samsung.ecomm 1600034861 https://www.samsung.com/us/account/privacy-policy/
   snapshot: 20241013073238
7 com.samsung.ecomm 1600034977 https://www.samsung.com/us/account/privacy-policy/
   snapshot: 20241013073238
url already replicated, skipping download.
8 com.saphir.baridbankmobile 609008 nan
9 com.saphir.baridbankmobile 609009 nan
10 com.satfinder.dishtv.satelittefinder.ar.dishalign.setdishtv 23 nan
11 com.satfinder.dishtv.satelittefinder.ar.dishalign.setdishtv 24 nan
12 com.satispay.customer 9418 nan
13 com.satispay.customer 9421 nan
14 com.saunalahti.oma 960 https://safeavenue.f-secure.com/iframe/-sso/saunalahti/?lang=fi&route=privacy
   no snapshot
15 com.saunalahti.oma 965 https://safeavenue.f-secure.com/iframe/-sso/saunalahti/?lang=fi&route=privacyA_ID_

Update the table

Then download the Markdown files. The download results are stored in 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\pp_wayback_result.csv

For successful downloads, update the source field in apk_versions_summary.csv from apkitself to apkitself_waybackmachine for the row with the corresponding apkname and version.

In [174]:
import re
import shutil
from pathlib import Path
import pandas as pd
from __future__ import annotations
from datetime import datetime

In [175]:
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch\apk_versions_summary.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_gpt.csv"
maintainess_apk_summary_df = pd.read_csv(apk_summary_df_path)
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                    apk_name    version  \
 0       com.S2.WaStickerApps         33   
 1       com.S2.WaStickerApps         34   
 2     com.Slbazer.Craftsman4  972000073   
 3     com.Slbazer.Craftsman4  972000074   
 4  com.rubygames.splashyjump       3902   
 
                                     json_file source privacy_url source_file  
 0           com.S2.WaStickerApps-33_urls.json    NaN         NaN         NaN  
 1           com.S2.WaStickerApps-34_urls.json    NaN         NaN         NaN  
 2  com.Slbazer.Craftsman4-972000073_urls.json    NaN         NaN         NaN  
 3  com.Slbazer.Craftsman4-972000074_urls.json    NaN         NaN         NaN  
 4    com.rubygames.splashyjump-3902_urls.json    NaN         NaN         NaN  ,
 (100, 6))

In [176]:
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\waybackmachine\pp_wayback_result.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\waybackmachine\pp_wayback_result.csv
csv_path = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\waybackmachine\pp_wayback_result.csv")
md_results_df = pd.read_csv(csv_path, dtype={"apk_name": str, "version": str})
md_results_df.head(), md_results_df.shape

(                    apk_name    version pp_url  timestamp wayback_url  \
 0       com.S2.WaStickerApps         33    NaN        NaN         NaN   
 1       com.S2.WaStickerApps         34    NaN        NaN         NaN   
 2     com.Slbazer.Craftsman4  972000073    NaN        NaN         NaN   
 3     com.Slbazer.Craftsman4  972000074    NaN        NaN         NaN   
 4  com.rubygames.splashyjump       3902    NaN        NaN         NaN   
 
    has_snapshot  downloaded  
 0         False       False  
 1         False       False  
 2         False       False  
 3         False       False  
 4         False       False  ,
 (98, 7))

In [177]:
# 0. Make a copy first to avoid modifying the original table
left = maintainess_apk_summary_df.copy()
right = md_results_df.copy()
left.head(), right.head()

(                    apk_name    version  \
 0       com.S2.WaStickerApps         33   
 1       com.S2.WaStickerApps         34   
 2     com.Slbazer.Craftsman4  972000073   
 3     com.Slbazer.Craftsman4  972000074   
 4  com.rubygames.splashyjump       3902   
 
                                     json_file source privacy_url source_file  
 0           com.S2.WaStickerApps-33_urls.json    NaN         NaN         NaN  
 1           com.S2.WaStickerApps-34_urls.json    NaN         NaN         NaN  
 2  com.Slbazer.Craftsman4-972000073_urls.json    NaN         NaN         NaN  
 3  com.Slbazer.Craftsman4-972000074_urls.json    NaN         NaN         NaN  
 4    com.rubygames.splashyjump-3902_urls.json    NaN         NaN         NaN  ,
                     apk_name    version pp_url  timestamp wayback_url  \
 0       com.S2.WaStickerApps         33    NaN        NaN         NaN   
 1       com.S2.WaStickerApps         34    NaN        NaN         NaN   
 2     com.Slbazer.Craftsman4  

In [178]:
# 1. Normalize the types of key fields
for df in [left, right]:
    df["apk_name"] = df["apk_name"].astype(str).str.strip()
    df["version"] = df["version"].astype(str).str.strip()

In [179]:
# 2. Keep only rows where downloadedwei is true and wayback_url is not empty
src = right.loc[
    right["downloaded"] &
    (right["wayback_url"].astype(str).str.strip() != ""),
    ["apk_name", "version", "pp_url", "wayback_url", "downloaded"]
].copy()
src.head(), src.shape

(               apk_name     version  \
 6     com.samsung.ecomm  1600034861   
 7     com.samsung.ecomm  1600034977   
 18       com.scn.twok48          50   
 19       com.scn.twok48          51   
 24  com.screenmirrorapp         307   
 
                                                pp_url  \
 6   https://www.samsung.com/us/account/privacy-pol...   
 7   https://www.samsung.com/us/account/privacy-pol...   
 18  https://rawgit.com/ScNSpeed/ScN/master/Privacy...   
 19  https://rawgit.com/ScNSpeed/ScN/master/Privacy...   
 24          https://zipoapps.com/screenmirror/privacy   
 
                                           wayback_url  downloaded  
 6   https://web.archive.org/web/20241013073238/htt...        True  
 7   https://web.archive.org/web/20241013073238/htt...        True  
 18  https://web.archive.org/web/20241221123017/htt...        True  
 19  https://web.archive.org/web/20241221123017/htt...        True  
 24  https://web.archive.org/web/20240920032912/htt...        T

In [180]:
# 4. Left join to the target table
merged = left.merge(
    src,
    on=["apk_name", "version"],
    how="left",
    suffixes=("", "_new")
)
merged.head(), merged.shape

(                    apk_name    version  \
 0       com.S2.WaStickerApps         33   
 1       com.S2.WaStickerApps         34   
 2     com.Slbazer.Craftsman4  972000073   
 3     com.Slbazer.Craftsman4  972000074   
 4  com.rubygames.splashyjump       3902   
 
                                     json_file source privacy_url source_file  \
 0           com.S2.WaStickerApps-33_urls.json    NaN         NaN         NaN   
 1           com.S2.WaStickerApps-34_urls.json    NaN         NaN         NaN   
 2  com.Slbazer.Craftsman4-972000073_urls.json    NaN         NaN         NaN   
 3  com.Slbazer.Craftsman4-972000074_urls.json    NaN         NaN         NaN   
 4    com.rubygames.splashyjump-3902_urls.json    NaN         NaN         NaN   
 
   pp_url wayback_url downloaded  
 0    NaN         NaN        NaN  
 1    NaN         NaN        NaN  
 2    NaN         NaN        NaN  
 3    NaN         NaN        NaN  
 4    NaN         NaN        NaN  ,
 (100, 9))

In [181]:
# 6. Update the matched rows
# mask = merged["downloaded"] == True
# mask.value_counts()

In [182]:
# 6. Update the matched rows
mask = merged["downloaded"] == True

merged.loc[mask, "source"] = "apkitself_waybackmachine"
merged.head(), merged.shape

(                    apk_name    version  \
 0       com.S2.WaStickerApps         33   
 1       com.S2.WaStickerApps         34   
 2     com.Slbazer.Craftsman4  972000073   
 3     com.Slbazer.Craftsman4  972000074   
 4  com.rubygames.splashyjump       3902   
 
                                     json_file source privacy_url source_file  \
 0           com.S2.WaStickerApps-33_urls.json    NaN         NaN         NaN   
 1           com.S2.WaStickerApps-34_urls.json    NaN         NaN         NaN   
 2  com.Slbazer.Craftsman4-972000073_urls.json    NaN         NaN         NaN   
 3  com.Slbazer.Craftsman4-972000074_urls.json    NaN         NaN         NaN   
 4    com.rubygames.splashyjump-3902_urls.json    NaN         NaN         NaN   
 
   pp_url wayback_url downloaded  
 0    NaN         NaN        NaN  
 1    NaN         NaN        NaN  
 2    NaN         NaN        NaN  
 3    NaN         NaN        NaN  
 4    NaN         NaN        NaN  ,
 (100, 9))

In [183]:
# 7. Remove temporary columns
merged = merged.drop(columns=["pp_url"])
merged.head(), merged.shape

(                    apk_name    version  \
 0       com.S2.WaStickerApps         33   
 1       com.S2.WaStickerApps         34   
 2     com.Slbazer.Craftsman4  972000073   
 3     com.Slbazer.Craftsman4  972000074   
 4  com.rubygames.splashyjump       3902   
 
                                     json_file source privacy_url source_file  \
 0           com.S2.WaStickerApps-33_urls.json    NaN         NaN         NaN   
 1           com.S2.WaStickerApps-34_urls.json    NaN         NaN         NaN   
 2  com.Slbazer.Craftsman4-972000073_urls.json    NaN         NaN         NaN   
 3  com.Slbazer.Craftsman4-972000074_urls.json    NaN         NaN         NaN   
 4    com.rubygames.splashyjump-3902_urls.json    NaN         NaN         NaN   
 
   wayback_url downloaded  
 0         NaN        NaN  
 1         NaN        NaN  
 2         NaN        NaN  
 3         NaN        NaN  
 4         NaN        NaN  ,
 (100, 8))

In [184]:
# 8. Save back to CSV
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_waybackmachine_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch\apk_versions_summary_waybackmachine_md.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_waybackmachine_md.csv"
merged.to_csv(apk_summary_df_path, index=False)